# Width-256 FineWeb sweep
Select a BF16-capable GPU runtime (such as A100) and run the three cells in order. The first cell clones the project on a fresh runtime and prints the commit. A private repository requires Git credentials in Colab.

In [ ]:
from pathlib import Path
import subprocess

repo = Path("/content/cs182-final-proj")
if not repo.exists():
    subprocess.run(["git", "clone", "--branch", "adit-restructure",
                    "https://github.com/adit-yaya/cs182-final-proj.git", str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "rev-parse", "HEAD"], check=True)
assert (repo / "results/w256_bpe/tokenizer.json").is_file(), "Push the shared tokenizer to the branch first."

In [ ]:
import os
import sys
from google.colab import drive

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt"),
                "datasets==5.2.0", "huggingface_hub==2.4.0", "tiktoken==0.14.0", "tokenizers==0.23.3"], check=True)
drive.mount("/content/drive")

In [ ]:
import torch
sys.path.insert(0, str(repo / "scripts"))
from run_w256 import NS_PE, TIKH, run

assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), "Select a BF16-capable GPU."
family = NS_PE  # Change to TIKH for the four lambda settings.
run(family, create_tokenizer=False, device="cuda",
    out=Path("/content/drive/MyDrive/cs182-w256-colab"))